# Tercera Pre-Entrega (Checkpoint 1): Perfilado y Saneamiento Inicial
**Alumno:** Lautaro Suárez Silva  
**Proyecto:** Valuación de Transferencias de Futbolistas en las 5 Grandes Ligas Europeas (*Big 5*)  
**Datasets:** dataset_futbol_v2.csv (Consolidación de registros de Transfermarkt) + gdp-per-capita-worldbank-selected-columns.csv (PBI per cápita del Banco Mundial)


Para este trabajo, si bien no lo especifica la consigna, doy por entendido que es lo mismo usar Google Colab, tal y como vengo haciendo desde el principio (consulté con teacher y al parecer esta todo OK). Mis datasets esta en formato CSV. En la anterior entrega ya había sacado alguanas conclusiones sobre posbiles problemáticas en datos faltantes y cosas a corregir, pero vamos a tratar de hacerlo un poquito mejor para esta.

In [6]:
import pandas as pd
import numpy as np
import csv

# Cargo el dataset limpio v2
df = pd.read_csv("dataset_futbol_v2.csv")

# Convertimos la fecha de transferencia a formato datetime para que quede correctamente tipada, tengo entendido que esto lo voy a tener que hacer en todos los entornos que cree de ahora en más (?)
df['transfer_date'] = pd.to_datetime(df['transfer_date'])

# Inspección visual de los primeros registros
print("Primeros 5 Registros con .head()")
display(df.head())

# Volumen de datos y estructura de columnas
print(f"\nDimensiones exactas con .shape: {df.shape[0]} filas y {df.shape[1]} columnas.\n")
print("Información de columnas y tipos de datos con .info()")
df.info()

Primeros 5 Registros con .head()


,player_id,player_name,transfer_date,selling_club_name,goals_prior_season,assists_prior_season,minutes_played_prior_season,national_team_caps,player_age,contract_left_months,player_position,foot,buying_club_league,destination_gdp_per_capita,buying_club_revenue,transfer_fee_eur,transfer_fee_millions
0,164771,Jordan Veretout,2024-09-04,Marseille,4.0,4.0,3388.0,6.0,31.5,24,Midfield,right,FR1,55441.188,55441.19,4000000.0,4.0
1,211474,Ali Abdi,2024-09-02,SM Caen,0.0,0.0,0.0,49.0,30.7,24,Defender,left,FR1,55441.188,55441.19,500000.0,0.5
2,110864,Sam Johnstone,2024-08-30,Crystal Palace,0.0,0.0,2048.0,4.0,31.4,24,Goalkeeper,right,GB1,54542.016,98175.63,11900000.0,11.9
3,160668,Antonio Colak,2024-08-30,Parma,0.0,0.0,0.0,3.0,31.0,24,Attack,right,IT1,53312.070,58643.28,2500000.0,2.5
4,251302,Niklas Dorsch,2024-08-30,Augsburg,0.0,0.0,1449.0,13.0,26.6,24,Midfield,right,L1,63578.140,82651.58,3500000.0,3.5



Dimensiones exactas con .shape: 5290 filas y 17 columnas.

Información de columnas y tipos de datos con .info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5290 entries, 0 to 5289
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   player_id                    5290 non-null   int64         
 1   player_name                  5290 non-null   object        
 2   transfer_date                5290 non-null   datetime64[ns]
 3   selling_club_name            5290 non-null   object        
 4   goals_prior_season           5290 non-null   float64       
 5   assists_prior_season         5290 non-null   float64       
 6   minutes_played_prior_season  5290 non-null   float64       
 7   national_team_caps           4462 non-null   float64       
 8   player_age                   5288 non-null   float64       
 9   contract_left_months         5290 non-null   int64         


In [11]:
# Calculo el total y los porcentaje de valores nulos por columna
nulos_df = pd.DataFrame({
    'Total_Nulos': df.isnull().sum(),
    'Porcentaje_Nulos (%)': (df.isnull().mean() * 100).round(2)
})

print("Perfilado de valores nulos")
# Aca hice un pequeño filtro para mostrar solo las columnas que presentan al menos un nulo, para mayor claridad
display(nulos_df[nulos_df['Total_Nulos'] > 0])

# Estadísticas descriptivas de variables numéricas con .describe()
print("\nEstadística descriptiva de las variables numéricas")
display(df.describe().T.round(2))

# Estadísticas descriptivas de variables categóricas
print("\nEstadística descriptiva de las variables categóricas")
display(df.describe(include=['object']).T)

Perfilado de valores nulos


,Total_Nulos,Porcentaje_Nulos (%)
national_team_caps,828,15.65
player_age,2,0.04
foot,6,0.11



Estadística descriptiva de las variables numéricas


,count,mean,min,25%,50%,75%,max,std
player_id,5290.0,317838.285633,7161.0,149716.0,277635.0,429853.0,1193867.0,214255.610674
transfer_date,5290,2019-12-08 07:47:23.251417856,2012-01-01 00:00:00,2017-07-01 00:00:00,2020-07-01 00:00:00,2022-09-01 00:00:00,2024-09-04 00:00:00,NaN
goals_prior_season,5290.0,2.370321,0.0,0.0,0.0,2.0,50.0,4.842569
assists_prior_season,5290.0,1.593384,0.0,0.0,0.0,2.0,32.0,2.914758
minutes_played_prior_season,5290.0,1196.953875,0.0,0.0,695.5,2381.0,4848.0,1297.704005
national_team_caps,4462.0,27.306141,0.0,4.0,14.0,42.0,233.0,31.527559
player_age,5288.0,23.774981,16.4,21.4,23.4,25.7,36.6,3.3167
contract_left_months,5290.0,24.0,24.0,24.0,24.0,24.0,24.0,0.0
destination_gdp_per_capita,5290.0,53088.858136,40409.99,49039.57,53312.07,54856.867,63578.14,5525.465741
buying_club_revenue,5290.0,70983.470486,50610.63,55361.45,60687.22,87581.1,98649.8,17702.955644



Estadística descriptiva de las variables categóricas


,count,unique,top,freq
player_name,5290,3505,Gianluca Caprari,7
selling_club_name,5290,946,Juventus,50
player_position,5290,4,Attack,1694
foot,5284,3,right,3766
buying_club_league,5290,5,GB1,1404


In [13]:
# Eliminación de columnas innecesarias (.drop())
# Descartamos contract_left_months por tener varianza cero. El valor de todas las transferencias esta constante en 24, como volvimos a ver en la estadistica descriptiva de la celda anterior, y player_id por ser un ID sin valor analítico
columnas_a_eliminar = ['contract_left_months', 'player_id']
df_limpio = df.drop(columns=columnas_a_eliminar)

print(f"Columnas eliminadas: {columnas_a_eliminar}")
print(f"Dimensiones tras eliminar columnas: {df_limpio.shape}\n")

# Aplique este filtros de booleanos para construir un subconjunto analítico relevante (¿Cuántos jugadores de los que fueron vendidos tuvieron un rodaje real e "hiciron valer" su transferencia?)
# Filtro las transferencias de futbolistas con rodaje real (> 0 minutos previos) y monto >= 1 millón de euros
condicion_minutos = df_limpio['minutes_played_prior_season'] > 0
condicion_monto = df_limpio['transfer_fee_millions'] >= 1.0

df_subconjunto = df_limpio[condicion_minutos & condicion_monto].copy()

print(f"Registros en el dataset completo: {len(df_limpio)}")
print(f"Registros en el subconjunto filtrado (minutos > 0 y traspaso >= 1M EUR): {len(df_subconjunto)}")
print(f"Porcentaje de jugadores que cumplieron con el subconjunto: {(len(df_subconjunto) / len(df_limpio) * 100):.2f}%\n")

display(df_subconjunto.head())

Columnas eliminadas: ['contract_left_months', 'player_id']
Dimensiones tras eliminar columnas: (5290, 15)

Registros en el dataset completo: 5290
Registros en el subconjunto filtrado (minutos > 0 y traspaso >= 1M EUR): 2865
Porcentaje de jugadores que cumplieron con el subconjunto: 54.16%



,player_name,transfer_date,selling_club_name,goals_prior_season,assists_prior_season,minutes_played_prior_season,national_team_caps,player_age,player_position,foot,buying_club_league,destination_gdp_per_capita,buying_club_revenue,transfer_fee_eur,transfer_fee_millions
0,Jordan Veretout,2024-09-04,Marseille,4.0,4.0,3388.0,6.0,31.5,Midfield,right,FR1,55441.188,55441.19,4000000.0,4.0
2,Sam Johnstone,2024-08-30,Crystal Palace,0.0,0.0,2048.0,4.0,31.4,Goalkeeper,right,GB1,54542.016,98175.63,11900000.0,11.9
4,Niklas Dorsch,2024-08-30,Augsburg,0.0,0.0,1449.0,13.0,26.6,Midfield,right,L1,63578.140,82651.58,3500000.0,3.5
6,Mikey Johnston,2024-08-30,Celtic,2.0,2.0,426.0,19.0,25.4,Attack,right,GB1,54542.016,98175.63,3500000.0,3.5
8,Scott McTominay,2024-08-30,Man Utd,10.0,3.0,2775.0,73.0,27.7,Midfield,right,IT1,53312.070,58643.28,30500000.0,30.5


## Reflexión final y diagnóstico de los datos

Al revisar en detalle el dataset me encontré con varias cuestiones que vale la pena destacar antes de pasar a la etapa de modelado. Lo primero y más evidente, que ya había saltado en la entrega anterior, fue el problema con contract_left_months: todos los registros figuran con 24 meses restantes de contrato (desvío estándar en cero). Al chequear las tablas originales vi que no existe el dato histórico de vencimiento de contrato al momento de cada pase, por lo que ese 24 quedó como un valor fijo de relleno. Como una variable sin variabilidad no le aporta nada al análisis, decidí eliminarla junto con player_id, que es solo un identificador que no creo que lo use.

Otra cosa que me llamó la atención es que el 41,27% de los jugadores tiene 0 minutos jugados en la temporada anterior. Al principio parece raro que casi la mitad de los transferidos no haya tenido rodaje, pero cruzando la lógica del mercado con la estructura de los datos tiene sentido: además de los juveniles de reserva o jugadores que son transferidos por falta de continuidad, el archivo appearances.csv (uno de los originales que descargue de Kaggle) cubre principalmente las 5 grandes ligas europeas. Entonces, si un club ficha a un jugador desde Sudamérica, de una segunda división o de una liga menor, sus partidos no figuran en el registro y la suma queda en cero. Con el filtro booleano (exigiendo minutos previos mayores a cero y montos desde 1 millón de euros) pude aislar el subconjunto de operaciones que sí cuentan con historial deportivo comprobable (esos tipos de transfrencias para las cuales pretendo armar el modelo).

Respecto a los valores faltantes, el único volumen importante está en national_team_caps (828 nulos, alrededor del 15,65%). Como el dataset sí distingue a quienes tienen 0 partidos internacionales (232 casos), interpreto que estos nulos corresponden a jugadores sin historial registrado en selecciones mayores. En cambio, los nulos de foot (6) y player_age (2) son marginales y no afectan la muestra. Por último, buying_club_revenue (variable que representa el poder de compra del club comprador) está calculada actualmente como un múltiplo del PBI per cápita según la liga de destino. Me parece un "proxy" medio básico comparado a lo que había expuesto en la presentación de mi proyecto en la primera pre-entrega, así que más adelante la voy a reemplazar, si se puede, por una métrica real del poder de compra de cada club (como su gasto histórico en fichajes o el valor de mercado del plantel, solo si es que existen estos datos y no estoy tirando cualquiera).

De cara al modelo predictivo, mi variable objetivo va a ser transfer_fee_millions. Para explicarla, las variables clave van a ser las de rendimiento (minutes_played_prior_season, goals_prior_season, assists_prior_season y national_team_caps), las del perfil del jugador (player_age, que pesa mucho por la proyección de reventa, y player_position) y las de contexto económico (destination_gdp_per_capita y buying_club_league, para capturar la diferencia de billetera entre la Premier League y el resto de las ligas).